# Validation suite — post-`5_organize` pipeline output

Exhaustive sanity checks on `ltm_grided_clipped_A_D_Z_Y_SelfPipeline.parquet`.

## Schema (verified against the actual file)

- **`k`** (gridding index from `3_organize`): integer in $[1, 60]$
- **`t`** (decision index from `5_organize` `cumcount()`): integer in $[0, 59]$
- **`k = t + 1`** identically across all rows
- $\text{grid\_end}(t) = (t+1) \cdot 12\text{h} = k \cdot 12\text{h}$ from intime
- $t = 0$ corresponds to the first decision at $\tau_0 = \text{intime} + 12\text{h}$
- $t = 59$ corresponds to the last decision at $\tau_0 + 29 \cdot 12\text{h} = \text{intime} + 720\text{h}$
- $W^L_t = [\text{grid\_end}(t) - 12,\ \text{grid\_end}(t))$, right-open
- $K = \tau_0 + 30\text{d} = \text{intime} + 732\text{h}$
- Eligibility: $\text{los} \geq 17\text{h}$ AND $\text{death\_time\_from\_intime}$ is NaN or $\geq 12\text{h}$
- Leakage buffer: no observation within 5h of discharge

## Outcome encoding contract

| Terminal type     | $A$ | $D$   | $Z$    | $Y$ |
|-------------------|-----|-------|--------|-----|
| In-ICU death      | 0   | 1     | NaN    | 1   |
| Discharged        | 1   | NaN   | {0,1}  | $Z$ |
| Admin-censored    | 0   | 0     | NaN    | 0   |
| Non-terminal rows | 0   | 0     | NaN    | NaN |

## Outputs

- `validation_outputs/validation_report.csv` — one row per test
- `validation_outputs/validation_report.json` — structured, with samples
- `validation_outputs/validation_summary.txt` — human-readable
- supporting CSVs for missingness, per-t statistics, implausibility

## Test categories

| Category | What it catches |
|---|---|
| **A. Schema**          | Column presence, dtypes, key uniqueness |
| **B. Grid integrity**  | $k/t$ contiguity, grid arithmetic, row counts |
| **C. Cohort eligibility** | Upstream filters (LOS, death timing) |
| **D. Leakage**         | 5h pre-discharge buffer respected |
| **E. Outcome encoding** | $(A,D,Z,Y)$ contract |
| **F. Cross-source consistency** | label vs derived; redundant fact agreement |
| **G. Distributional**  | Plausibility ranges, missingness patterns |

A, B, D, E are **hard contracts**. C, F, G are **informative**.


In [1]:
import pandas as pd
import numpy as np
import json
from datetime import datetime
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [2]:
# ---- Configuration ----
PARQUET_PATH = "../Dataset/mimic-iv/from_pipeline/ltm_grided_clipped_A_D_Z_Y_SelfPipeline.parquet"
OUTPUT_DIR   = Path("./validation_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Schema constants
W_L_HOURS         = 12
GRID_STEP_HOURS   = 12
LEAK_BUFFER_H     = 5
MIN_USABLE_HOURS  = 12
MIN_LOS_HOURS     = MIN_USABLE_HOURS + LEAK_BUFFER_H        # 17h
TAU_0_OFFSET_H    = 12
HORIZON_DAYS      = 30
K_HOURS           = TAU_0_OFFSET_H + HORIZON_DAYS * 24      # 732h
T_MAX_EXPECTED    = 59
T_MIN_EXPECTED    = 0
K_INDEX_MIN       = 1
K_INDEX_MAX       = 60

UNKNOWN_TOKEN     = "unknown"

# Confounder column groups (real names from the parquet).
MEAN_VARS = [
    "temperature__mean__last_12h",
    "heart_rate__mean__last_12h",
    "arterial_blood_pressure_mean__mean__last_12h",
    "fluid_out_urine__mean__last_12h",
    "pco2_arterial__mean__last_12h",
    "respiratory_rate_measured__mean__last_12h",
    "o2_saturation__mean__last_12h",
    "po2_arterial__mean__last_12h",
]
LAST_VARS_NUMERIC = [
    "bicarbonate_arterial__last__last_12h",
    "activated_partial_thromboplastin_time__last__last_12h",
    "hemoglobin__last__last_12h",
    "creatinine__last__last_12h",
    "ureum__last__last_12h",
    "lactate__last__last_12h",
]
LAST_VARS_CATEGORICAL = [
    "glasgow_coma_scale_total__last__last_12h",
    "fio2__last__last_12h",
    "vent_mode__last__last_12h",
]
VENT_HOURS_COL = "vent_mode__hours_since_last__last_12h"
ALL_L_COLS = MEAN_VARS + LAST_VARS_NUMERIC + LAST_VARS_CATEGORICAL + [VENT_HOURS_COL]


In [3]:
df = pd.read_parquet(PARQUET_PATH)
print(f"Loaded: {len(df):,} rows, {df['stay_id'].nunique():,} stays.")
print(f"Columns: {len(df.columns)}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")


Loaded: 557,896 rows, 81,736 stays.
Columns: 42
Memory: 0.31 GB


In [4]:
# Convenience columns and dtype normalisation.
df["_grid_end_h"] = df["grid_end"].dt.total_seconds() / 3600.0
df["_los_h"]      = df["los"].dt.total_seconds() / 3600.0
for col in ("intime", "outtime", "death_abs_time"):
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Terminal / non-terminal masks.
df_sorted = df.sort_values(["stay_id", "t"])
last_idx = df_sorted.groupby("stay_id", sort=False).tail(1).index
term = df.loc[last_idx].copy()
non_term = df.loc[~df.index.isin(last_idx)].copy()
print(f"Terminal rows: {len(term):,}")
print(f"Non-terminal rows: {len(non_term):,}")

# Per-stay summaries.
los_per_stay     = df.groupby("stay_id")["_los_h"].first()
death_per_stay   = df.groupby("stay_id")["death_time_from_intime"].first()
intime_per_stay  = df.groupby("stay_id")["intime"].first()
outtime_per_stay = df.groupby("stay_id")["outtime"].first()


Terminal rows: 81,736
Non-terminal rows: 476,160


## Test framework

`record()` appends one row to `report_rows`. After all tests, results are
compiled into `report_df` and exported to disk.

In [5]:
report_rows = []

def record(category, test_id, description, passed, n_violations=0,
           n_total=None, details="", sample=None):
    if sample is None:
        sample = []
    elif hasattr(sample, "tolist"):
        sample = sample.tolist()
    sample = [str(s) for s in sample[:5]]
    report_rows.append({
        "category": category,
        "test_id": test_id,
        "description": description,
        "passed": bool(passed),
        "n_violations": int(n_violations),
        "n_total_checked": int(n_total) if n_total is not None else None,
        "violation_rate": (n_violations / n_total) if (n_total and n_total > 0) else None,
        "details": details,
        "sample_violating_stay_ids": sample,
    })

def show_last():
    r = report_rows[-1]
    flag = "PASS" if r["passed"] else "FAIL"
    extra = f" ({r['n_violations']}/{r['n_total_checked']})" if r['n_total_checked'] else ""
    print(f"[{flag}] [{r['category']}/{r['test_id']}] {r['description']}{extra}")
    if not r["passed"]:
        if r["sample_violating_stay_ids"]:
            print(f"        sample: {r['sample_violating_stay_ids']}")
        if r["details"]:
            print(f"        details: {r['details']}")
    elif r["details"]:
        print(f"        info: {r['details']}")


## A. Schema tests

Column presence, dtypes, key uniqueness, value domains.

In [6]:
# A1: required columns present.
REQUIRED_COLS = {
    "stay_id", "t", "k", "grid_end", "intime", "outtime", "los",
    "A", "D", "Z", "Y",
    "icu_mortality", "icu_mortality_flag",
    "mortality_after_discharge_30d",
    "death_time_from_intime",
    "subject_id", "raw_age", "sex", "unit_type", "origin",
}
missing = REQUIRED_COLS - set(df.columns)
record("A.Schema", "A1", "All required columns present",
       passed=len(missing) == 0,
       details=f"Missing: {sorted(missing)}" if missing else "OK")
show_last()


[PASS] [A.Schema/A1] All required columns present
        info: OK


In [7]:
# A2: (stay_id, t) is a unique key.
dup_t = df.duplicated(subset=["stay_id", "t"], keep=False)
record("A.Schema", "A2", "(stay_id, t) is a unique key",
       passed=not dup_t.any(),
       n_violations=int(dup_t.sum()),
       n_total=len(df),
       sample=df.loc[dup_t, "stay_id"].drop_duplicates())
show_last()

# A2b: (stay_id, k) is also a unique key (since k = t + 1).
dup_k = df.duplicated(subset=["stay_id", "k"], keep=False)
record("A.Schema", "A2b", "(stay_id, k) is a unique key",
       passed=not dup_k.any(),
       n_violations=int(dup_k.sum()),
       n_total=len(df),
       sample=df.loc[dup_k, "stay_id"].drop_duplicates())
show_last()


[PASS] [A.Schema/A2] (stay_id, t) is a unique key (0/557896)
[PASS] [A.Schema/A2b] (stay_id, k) is a unique key (0/557896)


In [8]:
# A3: no NaN in key / structural columns.
key_cols = ["stay_id", "t", "k", "grid_end", "intime", "outtime", "los"]
nulls = {c: int(df[c].isna().sum()) for c in key_cols if c in df.columns}
total = sum(nulls.values())
record("A.Schema", "A3", "No NaN in stay_id, t, k, grid_end, intime, outtime, los",
       passed=total == 0,
       n_violations=total,
       details=str(nulls))
show_last()


[PASS] [A.Schema/A3] No NaN in stay_id, t, k, grid_end, intime, outtime, los
        info: {'stay_id': 0, 't': 0, 'k': 0, 'grid_end': 0, 'intime': 0, 'outtime': 0, 'los': 0}


In [9]:
# A4: t in [0, 59], k in [1, 60].
t_min, t_max = int(df["t"].min()), int(df["t"].max())
k_min, k_max = int(df["k"].min()), int(df["k"].max())
t_ok = (df["t"] >= T_MIN_EXPECTED) & (df["t"] <= T_MAX_EXPECTED)
k_ok = (df["k"] >= K_INDEX_MIN) & (df["k"] <= K_INDEX_MAX)
record("A.Schema", "A4", f"t in [{T_MIN_EXPECTED}, {T_MAX_EXPECTED}] and k in [{K_INDEX_MIN}, {K_INDEX_MAX}]",
       passed=t_ok.all() and k_ok.all(),
       n_violations=int((~t_ok).sum() + (~k_ok).sum()),
       n_total=len(df),
       details=f"t: [{t_min}, {t_max}]; k: [{k_min}, {k_max}]")
show_last()


[PASS] [A.Schema/A4] t in [0, 59] and k in [1, 60] (0/557896)
        info: t: [0, 59]; k: [1, 60]


In [10]:
# A4b: k == t + 1 everywhere.
diff = df["k"] - df["t"]
bad = diff != 1
record("A.Schema", "A4b", "k == t + 1 for every row",
       passed=not bad.any(),
       n_violations=int(bad.sum()),
       n_total=len(df),
       details=f"unique(k - t): {sorted(diff.unique())}")
show_last()


[PASS] [A.Schema/A4b] k == t + 1 for every row (0/557896)
        info: unique(k - t): [np.int64(1)]


In [11]:
# A5: A, D, Z, Y domains.
def _domain(s, allowed):
    return set(s.dropna().unique()) - set(allowed)

bad = {
    "A": _domain(df["A"], {0, 1}),
    "D": _domain(df["D"], {0.0, 1.0}),
    "Z": _domain(df["Z"], {0.0, 1.0}),
    "Y": _domain(df["Y"], {0.0, 1.0}),
}
total = sum(len(v) for v in bad.values())
record("A.Schema", "A5", "A in {0,1}; D, Z, Y in {0, 1, NaN}",
       passed=total == 0,
       n_violations=total,
       details=str({k: list(v) for k, v in bad.items() if v}))
show_last()


[PASS] [A.Schema/A5] A in {0,1}; D, Z, Y in {0, 1, NaN}
        info: {}


In [12]:
# A6: A is never NaN.
n_a_nan = int(df["A"].isna().sum())
record("A.Schema", "A6", "A is never NaN",
       passed=n_a_nan == 0,
       n_violations=n_a_nan,
       n_total=len(df))
show_last()


[PASS] [A.Schema/A6] A is never NaN (0/557896)


In [13]:
# A7: dtype expectations.
expected = {
    "stay_id": ["object", "string", "str"],
    "t": ["int"],
    "k": ["int"],
    "grid_end": ["timedelta64"],
    "intime": ["datetime64"],
    "outtime": ["datetime64"],
    "los": ["timedelta64"],
    "A": ["int"],
    "D": ["float"],
    "Z": ["float"],
    "Y": ["float"],
    "icu_mortality": ["bool"],
    "icu_mortality_flag": ["int"],
}
mm = {}
for col, kinds in expected.items():
    if col not in df.columns:
        mm[col] = "missing"
        continue
    actual_str = str(df[col].dtype).lower()
    if not any(k in actual_str for k in kinds):
        mm[col] = f"expected one of {kinds}, got {df[col].dtype}"
record("A.Schema", "A7", "Column dtypes match expectations",
       passed=len(mm) == 0,
       n_violations=len(mm),
       details=str(mm) if mm else "All match")
show_last()


[PASS] [A.Schema/A7] Column dtypes match expectations
        info: All match


## B. Grid integrity

Per-stay $t$ contiguity, grid_end arithmetic, row count rules.

In [14]:
# B1: per-stay t contiguous from 0.
def _contig_from_0(g):
    tv = np.sort(g["t"].values)
    return (tv == np.arange(len(tv))).all()

ok = df.groupby("stay_id").apply(_contig_from_0)
bad = ok[~ok].index
record("B.Grid", "B1", "Per-stay t is contiguous starting at 0",
       passed=bad.empty,
       n_violations=len(bad),
       n_total=df["stay_id"].nunique(),
       sample=bad)
show_last()


[PASS] [B.Grid/B1] Per-stay t is contiguous starting at 0 (0/81736)


/tmp/ipykernel_876697/3616301930.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ok = df.groupby("stay_id").apply(_contig_from_0)


In [15]:
# B2: grid_end_h == (t+1) * 12 (equivalently k * 12).
expected_ge_t = (df["t"] + 1) * GRID_STEP_HOURS
expected_ge_k = df["k"] * GRID_STEP_HOURS
bad_t = (df["_grid_end_h"] - expected_ge_t).abs() > 1e-6
bad_k = (df["_grid_end_h"] - expected_ge_k).abs() > 1e-6
record("B.Grid", "B2", "grid_end_h == (t+1) * 12 == k * 12",
       passed=not bad_t.any() and not bad_k.any(),
       n_violations=int(bad_t.sum()) + int(bad_k.sum()),
       n_total=2 * len(df),
       sample=df.loc[bad_t | bad_k, "stay_id"].drop_duplicates())
show_last()


[PASS] [B.Grid/B2] grid_end_h == (t+1) * 12 == k * 12 (0/1115792)


In [16]:
# B3: per-stay row count = min(floor((los-5)/12), T_MAX+1).
# T_MAX+1 = 60 because t goes from 0 to T_MAX (inclusive).
expected_n = np.minimum(
    np.floor((los_per_stay - LEAK_BUFFER_H) / GRID_STEP_HOURS).astype(int),
    T_MAX_EXPECTED + 1
)
actual_n = df.groupby("stay_id").size()
aligned = pd.DataFrame({"expected": expected_n, "actual": actual_n}).dropna()
mismatch = aligned[aligned["expected"] != aligned["actual"]]
record("B.Grid", "B3", "Per-stay row count = min(floor((los-5)/12), T_MAX+1)",
       passed=mismatch.empty,
       n_violations=len(mismatch),
       n_total=len(aligned),
       sample=mismatch.index,
       details="Mismatches indicate gridding-rule drift.")
show_last()


[PASS] [B.Grid/B3] Per-stay row count = min(floor((los-5)/12), T_MAX+1) (0/81736)
        info: Mismatches indicate gridding-rule drift.


In [17]:
# B4: grid_end strictly increasing within each stay.
ok = df.sort_values(["stay_id", "t"]).groupby("stay_id")["_grid_end_h"].apply(
    lambda s: (s.diff().dropna() > 0).all()
)
bad = ok[~ok].index
record("B.Grid", "B4", "grid_end strictly increasing within each stay",
       passed=bad.empty,
       n_violations=len(bad),
       n_total=df["stay_id"].nunique(),
       sample=bad)
show_last()


[PASS] [B.Grid/B4] grid_end strictly increasing within each stay (0/81736)


In [18]:
# B5: max grid_end across the dataset == T_MAX_EXPECTED + 1) * 12 = 720h.
max_ge_h = df["_grid_end_h"].max()
expected_max = (T_MAX_EXPECTED + 1) * GRID_STEP_HOURS
record("B.Grid", "B5", f"max grid_end == {expected_max}h",
       passed=max_ge_h == expected_max,
       details=f"observed max grid_end_h = {max_ge_h}")
show_last()


[PASS] [B.Grid/B5] max grid_end == 720h
        info: observed max grid_end_h = 720.0


## C. Cohort eligibility

LOS $\geq 17$h AND no death within first 12h.

In [19]:
# C1: los >= 17h.
short = los_per_stay[los_per_stay < MIN_LOS_HOURS].index
record("C.Cohort", "C1", f"los >= {MIN_LOS_HOURS}h for every stay",
       passed=short.empty,
       n_violations=len(short),
       n_total=len(los_per_stay),
       sample=short,
       details=f"min observed los: {los_per_stay.min():.2f}h")
show_last()


[PASS] [C.Cohort/C1] los >= 17h for every stay (0/81736)
        info: min observed los: 17.00h


In [20]:
# C2: no death before tau_0.
pb = death_per_stay.notna() & (death_per_stay < pd.Timedelta(hours=TAU_0_OFFSET_H))
pb_stays = death_per_stay[pb].index
record("C.Cohort", "C2", "No death before tau_0 (12h since intime)",
       passed=pb_stays.empty,
       n_violations=len(pb_stays),
       n_total=int(death_per_stay.notna().sum()),
       sample=pb_stays,
       details="If non-empty: 3_organize eligibility filter regressed.")
show_last()


[PASS] [C.Cohort/C2] No death before tau_0 (12h since intime) (0/31908)
        info: If non-empty: 3_organize eligibility filter regressed.


In [21]:
# C3: baseline columns constant within stay.
BASELINE_COLS = [
    "subject_id", "raw_age", "sex", "raw_height", "raw_weight",
    "unit_type", "origin", "intime", "outtime", "los",
    "death_time_from_intime", "icu_mortality", "icu_mortality_flag",
    "mortality_after_discharge_30d", "death_abs_time",
]
vbc = {}
for col in BASELINE_COLS:
    if col not in df.columns:
        continue
    n_var = df.groupby("stay_id")[col].nunique(dropna=False)
    n_bad = int((n_var > 1).sum())
    if n_bad > 0:
        vbc[col] = n_bad
record("C.Cohort", "C3", "Baseline columns constant within stay",
       passed=len(vbc) == 0,
       n_violations=sum(vbc.values()),
       details=str(vbc) if vbc else "All constant")
show_last()


[PASS] [C.Cohort/C3] Baseline columns constant within stay
        info: All constant


## D. Leakage tests

5h pre-discharge buffer respected. Admin-censored stays are exempt because
their grid is bounded by $K$, not by LOS.

In [22]:
# Admin-cens set.
admin_stays = set(term.loc[(term["A"] == 0) & (term["D"] == 0), "stay_id"])
non_admin_stays = set(df["stay_id"].unique()) - admin_stays
print(f"Admin-cens stays: {len(admin_stays):,}")
print(f"Non-admin stays:  {len(non_admin_stays):,}")


Admin-cens stays: 617
Non-admin stays:  81,119


In [23]:
# D1: for non-admin stays, max grid_end <= los - 5h.
max_ge = df.groupby("stay_id")["_grid_end_h"].max()
non_admin_idx = pd.Index(list(non_admin_stays))
diff = max_ge.loc[non_admin_idx] - (los_per_stay.loc[non_admin_idx] - LEAK_BUFFER_H)
viol = diff[diff > 1e-6].index
record("D.Leakage", "D1", "Non-admin stays: max grid_end <= los - 5h",
       passed=len(viol) == 0,
       n_violations=len(viol),
       n_total=len(non_admin_idx),
       sample=viol)
show_last()


[PASS] [D.Leakage/D1] Non-admin stays: max grid_end <= los - 5h (0/81119)


In [24]:
# D2: admin stays must have los > T_MAX*12 + leak = 725h.
admin_los = los_per_stay.loc[list(admin_stays)]
short_admin = admin_los[admin_los <= (T_MAX_EXPECTED + 1) * GRID_STEP_HOURS - GRID_STEP_HOURS + LEAK_BUFFER_H].index
# More transparent: admin stays satisfy n_rows = T_MAX + 1 = 60, meaning floor((los-5)/12) >= 60, so los >= 725h.
record("D.Leakage", "D2", "Admin-cens stays have los >= 725h",
       passed=short_admin.empty,
       n_violations=len(short_admin),
       n_total=len(admin_los),
       sample=short_admin,
       details=f"min los among admin-cens: {admin_los.min():.2f}h" if len(admin_los) else "no admin stays")
show_last()


[PASS] [D.Leakage/D2] Admin-cens stays have los >= 725h (0/617)
        info: min los among admin-cens: 737.08h


In [25]:
# D3: admin stays max grid_end == 720h.
expected_admin_max = (T_MAX_EXPECTED + 1) * GRID_STEP_HOURS
admin_max = max_ge.loc[list(admin_stays)]
viol = admin_max[admin_max != expected_admin_max].index
record("D.Leakage", "D3", f"Admin-cens stays have max grid_end == {expected_admin_max}h",
       passed=viol.empty,
       n_violations=len(viol),
       n_total=len(admin_max),
       sample=viol)
show_last()


[PASS] [D.Leakage/D3] Admin-cens stays have max grid_end == 720h (0/617)


## E. Outcome encoding contract

Non-terminal rows: A=0, D=0, Z=NaN, Y=NaN. Terminal rows: one of three
mutually exclusive (A, D, Z, Y) combinations.

In [26]:
# E1-E4: non-terminal rows.
for tid, col, expected, isnan in [
    ("E1", "A", 0, False),
    ("E2", "D", 0, False),
    ("E3", "Z", None, True),
    ("E4", "Y", None, True),
]:
    if isnan:
        bad = non_term[non_term[col].notna()]
        desc = f"Non-terminal {col} is NaN"
    else:
        bad = non_term[non_term[col] != expected]
        desc = f"Non-terminal {col} == {expected}"
    record("E.Encoding", tid, desc,
           passed=bad.empty,
           n_violations=len(bad),
           n_total=len(non_term),
           sample=bad["stay_id"].drop_duplicates())
    show_last()


[PASS] [E.Encoding/E1] Non-terminal A == 0 (0/476160)
[PASS] [E.Encoding/E2] Non-terminal D == 0 (0/476160)
[PASS] [E.Encoding/E3] Non-terminal Z is NaN (0/476160)
[PASS] [E.Encoding/E4] Non-terminal Y is NaN (0/476160)


In [27]:
# E5: terminal D is NaN iff A == 1.
bad = term[term["D"].isna() != term["A"].eq(1)]
record("E.Encoding", "E5", "Terminal: D is NaN iff A == 1",
       passed=bad.empty,
       n_violations=len(bad),
       n_total=len(term),
       sample=bad["stay_id"].drop_duplicates())
show_last()


[PASS] [E.Encoding/E5] Terminal: D is NaN iff A == 1 (0/81736)


In [28]:
# E6: terminal Z is non-NaN iff A == 1.
bad = term[term["Z"].notna() != term["A"].eq(1)]
record("E.Encoding", "E6", "Terminal: Z is non-NaN iff A == 1",
       passed=bad.empty,
       n_violations=len(bad),
       n_total=len(term),
       sample=bad["stay_id"].drop_duplicates())
show_last()


[PASS] [E.Encoding/E6] Terminal: Z is non-NaN iff A == 1 (0/81736)


In [29]:
# E7: terminal not (A==1 AND D==1).
absorbed = term["A"].eq(1).astype(int) + term["D"].eq(1).fillna(False).astype(int)
bad = term[absorbed > 1]
record("E.Encoding", "E7", "Terminal: not (A==1 AND D==1)",
       passed=bad.empty,
       n_violations=len(bad),
       n_total=len(term),
       sample=bad["stay_id"].drop_duplicates())
show_last()


[PASS] [E.Encoding/E7] Terminal: not (A==1 AND D==1) (0/81736)


In [30]:
# E8: terminal Y matches the (A, D, Z) rule.
def _expected_y(row):
    A, D, Z = row["A"], row["D"], row["Z"]
    if A == 0 and D == 1:
        return 1.0
    if A == 1:
        return float(Z) if pd.notna(Z) else np.nan
    if A == 0 and D == 0:
        return 0.0
    return np.nan

ey = term.apply(_expected_y, axis=1)
mismatch = (ey != term["Y"]) & ~(ey.isna() & term["Y"].isna())
bad = term.loc[mismatch]
record("E.Encoding", "E8", "Terminal Y is exactly determined by (A, D, Z)",
       passed=bad.empty,
       n_violations=len(bad),
       n_total=len(term),
       sample=bad["stay_id"].drop_duplicates())
show_last()


[PASS] [E.Encoding/E8] Terminal Y is exactly determined by (A, D, Z) (0/81736)


In [31]:
# E9: terminal cross-tab has only the 4 prescribed combos.
xt = term.groupby(["A", "D", "Z", "Y"], dropna=False).size().reset_index(name="n")
print("Terminal (A, D, Z, Y) cross-tab:")
print(xt.to_string(index=False))

def _norm(r):
    return tuple(None if pd.isna(v) else (int(v) if v in (0, 1) else float(v))
                 for v in (r["A"], r["D"], r["Z"], r["Y"]))

EXPECTED = [
    (0, 1.0, np.nan, 1.0),
    (1, np.nan, 0.0, 0.0),
    (1, np.nan, 1.0, 1.0),
    (0, 0.0, np.nan, 0.0),
]
expected_set = {tuple(None if pd.isna(v) else (int(v) if v in (0, 1) else float(v)) for v in c)
                for c in EXPECTED}
observed = {_norm(r) for _, r in xt.iterrows()}
unexpected = observed - expected_set
record("E.Encoding", "E9", "Terminal cross-tab has only the 4 prescribed combos",
       passed=len(unexpected) == 0,
       n_violations=len(unexpected),
       details=f"Unexpected: {unexpected}" if unexpected else "OK")
show_last()


Terminal (A, D, Z, Y) cross-tab:
 A   D   Z   Y     n
 0 0.0 NaN 0.0   617
 0 1.0 NaN 1.0  5330
 1 NaN 0.0 0.0 69788
 1 NaN 1.0 1.0  6001
[PASS] [E.Encoding/E9] Terminal cross-tab has only the 4 prescribed combos
        info: OK


## F. Cross-source consistency

Multiple columns encode overlapping facts: `icu_mortality` (timestamp-derived),
`icu_mortality_flag` (label), $D$, `mortality_after_discharge_30d`, $Z$, $Y$.
These should be coherent.

In [32]:
# F1: in-ICU death terminal rows have icu_mortality == True.
icu_death_term = term[(term["A"] == 0) & (term["D"] == 1)]
bad = icu_death_term[~icu_death_term["icu_mortality"].astype(bool)]
record("F.Consistency", "F1", "In-ICU death terminal rows have icu_mortality (derived) == True",
       passed=bad.empty,
       n_violations=len(bad),
       n_total=len(icu_death_term),
       sample=bad["stay_id"].drop_duplicates())
show_last()


[PASS] [F.Consistency/F1] In-ICU death terminal rows have icu_mortality (derived) == True (0/5330)


In [33]:
# F2: discharged terminal rows have icu_mortality == False.
disch_term = term[term["A"] == 1]
bad = disch_term[disch_term["icu_mortality"].astype(bool)]
record("F.Consistency", "F2", "Discharged terminal rows have icu_mortality (derived) == False",
       passed=bad.empty,
       n_violations=len(bad),
       n_total=len(disch_term),
       sample=bad["stay_id"].drop_duplicates())
show_last()


[PASS] [F.Consistency/F2] Discharged terminal rows have icu_mortality (derived) == False (0/75789)


In [34]:
# F3: admin-censored terminal rows are admin-censored *at K*, not necessarily
# alive forever. The legitimate cases:
#   - Patient died in ICU AFTER K  -> icu_mortality (derived) can be True;
#                                      death is past horizon, correctly excluded.
#   - Patient died in ICU at MIDNIGHT on the K-boundary day -> date-rounding artifact.
#
# Edge cases this test still flags (informational):
#   - Patient died in ICU before K but the stay extends past K (because
#     `outtime` is the bed-clear timestamp, post-mortem). 5_organize's
#     terminal encoder classifies these as admin-cens because LOS > K,
#     even though `death_time_from_intime <= K`. Strictly speaking the
#     terminal D should be 1 in this case. Whether to refine the encoder
#     to handle this depends on how many such cases there are (typically
#     a tiny minority).
K_per_stay_local = intime_per_stay + pd.Timedelta(hours=K_HOURS)
tau_0_per_stay   = intime_per_stay + pd.Timedelta(hours=TAU_0_OFFSET_H)
admin_term_local = term[(term["A"] == 0) & (term["D"] == 0)].copy()
admin_term_local["death_abs"]  = admin_term_local["stay_id"].map(
    df.groupby("stay_id")["death_abs_time"].first()
)
admin_term_local["K_time"]     = admin_term_local["stay_id"].map(K_per_stay_local)
admin_term_local["tau_0_time"] = admin_term_local["stay_id"].map(tau_0_per_stay)

# Candidate violations: death timestamp falls in (tau_0, K].
cand = admin_term_local[
    admin_term_local["death_abs"].notna()
    & (admin_term_local["death_abs"] > admin_term_local["tau_0_time"])
    & (admin_term_local["death_abs"] <= admin_term_local["K_time"])
].copy()

# Date-rounding artifact: death_abs is at midnight on the same calendar day as K.
cand["death_at_midnight"] = (cand["death_abs"].dt.time == pd.Timestamp("00:00:00").time())
cand["same_day_as_K"] = (cand["death_abs"].dt.normalize() == cand["K_time"].dt.normalize())
cand["likely_date_rounding"] = cand["death_at_midnight"] & cand["same_day_as_K"]

real_cand = cand[~cand["likely_date_rounding"]]
date_rounding = cand[cand["likely_date_rounding"]]

# Save the candidates for inspection.
if not cand.empty:
    cand[["stay_id", "death_abs", "tau_0_time", "K_time",
          "death_at_midnight", "same_day_as_K", "likely_date_rounding"]].to_csv(
        OUTPUT_DIR / "F3_candidates_admin_with_death_in_window.csv", index=False
    )

# Treat as informational — these are encoder edge cases, not strict contract violations.
record("F.Consistency", "F3", "Admin-cens stays with death in (tau_0, K]  (informational)",
       passed=True,
       n_violations=len(real_cand),
       n_total=len(admin_term_local),
       sample=real_cand["stay_id"].drop_duplicates(),
       details=(f"Real candidate edge cases: {len(real_cand)} "
                f"(LOS > K but death before K; encoder classifies as admin-cens). "
                f"Date-rounding artifacts: {len(date_rounding)}. "
                f"Candidates saved to F3_candidates_admin_with_death_in_window.csv. "
                f"Whether to refine the 5_organize encoder depends on count and impact."))
show_last()


[PASS] [F.Consistency/F3] Admin-cens stays with death in (tau_0, K]  (informational) (2/617)
        info: Real candidate edge cases: 2 (LOS > K but death before K; encoder classifies as admin-cens). Date-rounding artifacts: 3. Candidates saved to F3_candidates_admin_with_death_in_window.csv. Whether to refine the 5_organize encoder depends on count and impact.


In [35]:
# F4: Z == mortality_after_discharge_30d for discharged stays.
mismatch = disch_term[disch_term["Z"] != disch_term["mortality_after_discharge_30d"]]
record("F.Consistency", "F4", "Terminal Z == mortality_after_discharge_30d for discharged stays",
       passed=mismatch.empty,
       n_violations=len(mismatch),
       n_total=len(disch_term),
       sample=mismatch["stay_id"])
show_last()


[PASS] [F.Consistency/F4] Terminal Z == mortality_after_discharge_30d for discharged stays (0/75789)


In [36]:
# F5: label vs derived icu_mortality (informational; not a hard fail).
icu_d = term.set_index("stay_id")["icu_mortality"].astype(bool)
icu_l = term.set_index("stay_id")["icu_mortality_flag"].astype(bool)
xt2 = pd.crosstab(icu_l, icu_d,
                  rownames=["icu_mortality_flag (label)"],
                  colnames=["icu_mortality (derived)"])
print("Stay-level cross-tab:")
print(xt2)
n_label_only = int(((icu_l) & (~icu_d)).sum())
n_derived_only = int(((~icu_l) & (icu_d)).sum())
xt2.to_csv(OUTPUT_DIR / "label_vs_derived_icu_mortality.csv")

record("F.Consistency", "F5", "Label/derived icu_mortality disagreement (informational)",
       passed=True,
       n_violations=n_label_only + n_derived_only,
       details=(f"label_only (likely post-discharge in-hospital deaths): {n_label_only}; "
                f"derived_only (likely date-rounded timestamps): {n_derived_only}. "
                f"Cross-tab saved to label_vs_derived_icu_mortality.csv."))
show_last()


Stay-level cross-tab:
icu_mortality (derived)     False  True 
icu_mortality_flag (label)              
False                       73142     98
True                         3134   5362
[PASS] [F.Consistency/F5] Label/derived icu_mortality disagreement (informational)
        info: label_only (likely post-discharge in-hospital deaths): 3134; derived_only (likely date-rounded timestamps): 98. Cross-tab saved to label_vs_derived_icu_mortality.csv.


In [37]:
# F6: deaths in (outtime, K] yield terminal Z == 1.
K_per_stay = intime_per_stay + pd.Timedelta(hours=K_HOURS)
death_abs = df.groupby("stay_id")["death_abs_time"].first()
in_window = (
    death_abs.notna()
    & (death_abs > outtime_per_stay)
    & (death_abs <= K_per_stay)
)
in_window_stays = death_abs[in_window].index
z_lookup = term.set_index("stay_id")["Z"]
z_for_window = z_lookup.loc[in_window_stays]
bad = z_for_window[z_for_window != 1].index
record("F.Consistency", "F6", "Deaths in (outtime, K] have terminal Z == 1",
       passed=bad.empty,
       n_violations=len(bad),
       n_total=len(in_window_stays),
       sample=bad)
show_last()


[PASS] [F.Consistency/F6] Deaths in (outtime, K] have terminal Z == 1 (0/6001)


In [38]:
# F7: deaths after K should NOT yield terminal Z == 1.
after_K = death_abs.notna() & (death_abs > K_per_stay)
after_K_stays = death_abs[after_K].index
z_for_after = z_lookup.loc[after_K_stays]
bad = z_for_after[z_for_after == 1].index
record("F.Consistency", "F7", "Deaths after K have terminal Z != 1",
       passed=bad.empty,
       n_violations=len(bad),
       n_total=len(after_K_stays),
       sample=bad)
show_last()


[PASS] [F.Consistency/F7] Deaths after K have terminal Z != 1 (0/20572)


In [39]:
# F8: Y is exactly determined by (A, D, Z) — redundant cross-check from a different angle.
y_from_dz = np.where(
    term["A"].eq(0) & term["D"].eq(1), 1.0,
    np.where(term["A"].eq(1), term["Z"],
             np.where(term["A"].eq(0) & term["D"].eq(0), 0.0, np.nan))
)
mismatch = ~((y_from_dz == term["Y"]) | (pd.isna(y_from_dz) & term["Y"].isna()))
record("F.Consistency", "F8", "Y exactly determined by (A, D, Z) — independent check",
       passed=not mismatch.any(),
       n_violations=int(mismatch.sum()),
       n_total=len(term),
       sample=term.loc[mismatch, "stay_id"])
show_last()


[PASS] [F.Consistency/F8] Y exactly determined by (A, D, Z) — independent check (0/81736)


## G. Distributional checks (informational)

Plausibility ranges, rates, missingness patterns, per-$t$ statistics.
Most tests here are informational rather than hard pass/fail.

In [40]:
# G1: mortality / discharge rates.
n_icu_death = int(((term["A"] == 0) & (term["D"] == 1)).sum())
n_disch     = int((term["A"] == 1).sum())
n_admin     = int(((term["A"] == 0) & (term["D"] == 0)).sum())
n_total     = len(term)

rates = {
    "icu_death_rate": n_icu_death / n_total,
    "discharge_rate": n_disch / n_total,
    "admin_cens_rate": n_admin / n_total,
    "post_disch_mortality_among_discharged": disch_term["Z"].mean(),
    "overall_Y_rate": term["Y"].mean(),
}
for k, v in rates.items():
    print(f"  {k:50s} {v:.4f}")

PLAUSIBLE = {
    "icu_death_rate":                          (0.03, 0.20),
    "discharge_rate":                          (0.70, 0.97),
    "admin_cens_rate":                         (0.00, 0.05),
    "post_disch_mortality_among_discharged":   (0.02, 0.20),
    "overall_Y_rate":                          (0.05, 0.30),
}
implausible = {k: (v, PLAUSIBLE[k]) for k, v in rates.items()
               if not (PLAUSIBLE[k][0] <= v <= PLAUSIBLE[k][1])}
record("G.Distributional", "G1", "Mortality/discharge rates within plausible MIMIC-IV bands",
       passed=len(implausible) == 0,
       n_violations=len(implausible),
       details=str(implausible) if implausible else "All within plausible bands")
show_last()


  icu_death_rate                                     0.0652
  discharge_rate                                     0.9272
  admin_cens_rate                                    0.0075
  post_disch_mortality_among_discharged              0.0792
  overall_Y_rate                                     0.1386
[PASS] [G.Distributional/G1] Mortality/discharge rates within plausible MIMIC-IV bands
        info: All within plausible bands


In [41]:
# G2: terminal t distribution.
tt = term["t"]
print("Terminal t distribution:")
print(tt.describe().round(2).to_string())
print(f"\n% at T_MAX ({T_MAX_EXPECTED}): {(tt == T_MAX_EXPECTED).mean()*100:.4f}%")
print(f"% at t=0 (LOS just over 17h): {(tt == 0).mean()*100:.4f}%")

record("G.Distributional", "G2", "Terminal t distribution (informational)",
       passed=True,
       details=f"min={int(tt.min())}, median={int(tt.median())}, "
               f"mean={tt.mean():.2f}, max={int(tt.max())}, "
               f"pct_at_TMAX={(tt == T_MAX_EXPECTED).mean()*100:.2f}%")
show_last()


Terminal t distribution:
count    81736.00
mean         5.83
std          9.48
min          0.00
25%          1.00
50%          2.00
75%          6.00
max         59.00

% at T_MAX (59): 0.7842%
% at t=0 (LOS just over 17h): 22.0968%
[PASS] [G.Distributional/G2] Terminal t distribution (informational)
        info: min=0, median=2, mean=5.83, max=59, pct_at_TMAX=0.78%


In [42]:
# G3: rows per t (at-risk set; positivity diagnostic input).
n_per_t = df.groupby("t").size()
n_per_t.to_csv(OUTPUT_DIR / "rows_per_t.csv")

a_per_t = df.groupby("t")["A"].mean()
a_per_t.to_csv(OUTPUT_DIR / "discharge_rate_per_t.csv")

print("Rows per t (first/last 5):")
print(pd.concat([n_per_t.head(), n_per_t.tail()]).to_string())
print()
print("Discharge rate A=1 per t (first/last 5):")
print(pd.concat([a_per_t.head(), a_per_t.tail()]).round(4).to_string())

min_n = int(n_per_t.min())
bad = n_per_t[n_per_t < 100]
record("G.Distributional", "G3", "Per-t row count >= 100 (sufficient for propensity fitting)",
       passed=bad.empty,
       n_violations=len(bad),
       details=f"min n_per_t = {min_n}; per-t counts saved to rows_per_t.csv")
show_last()


Rows per t (first/last 5):
t
0     81736
1     63675
2     51483
3     39854
4     33590
55      767
56      741
57      695
58      676
59      641

Discharge rate A=1 per t (first/last 5):
t
0     0.2186
1     0.1840
2     0.2155
3     0.1474
4     0.1769
55    0.0274
56    0.0499
57    0.0187
58    0.0473
59    0.0312
[PASS] [G.Distributional/G3] Per-t row count >= 100 (sufficient for propensity fitting)
        info: min n_per_t = 641; per-t counts saved to rows_per_t.csv


In [43]:
# G4: L-column missingness by t.
present_L = [c for c in ALL_L_COLS if c in df.columns]
miss_overall = df[present_L].isna().mean().sort_values(ascending=False)
miss_by_t = df.groupby("t")[present_L].apply(lambda g: g.isna().mean())
miss_overall.to_csv(OUTPUT_DIR / "missingness_overall.csv")
miss_by_t.to_csv(OUTPUT_DIR / "missingness_by_t.csv")

print("Top-10 highest overall missingness:")
print(miss_overall.head(10).round(4).to_string())
print()
print("Missingness at t=0 (first 12h post-admit):")
print(miss_by_t.loc[0].sort_values(ascending=False).head(10).round(4).to_string())
print()
print(f"Missingness at t={T_MAX_EXPECTED}:")
if T_MAX_EXPECTED in miss_by_t.index:
    print(miss_by_t.loc[T_MAX_EXPECTED].sort_values(ascending=False).head(10).round(4).to_string())

very_missing = miss_overall[miss_overall > 0.5]
record("G.Distributional", "G4", "L-column missingness recorded; flag >50% missing",
       passed=very_missing.empty,
       n_violations=len(very_missing),
       details=(f"Columns with >50% missing: {list(very_missing.index)}. "
                "Saved missingness_overall.csv, missingness_by_t.csv.")
       if not very_missing.empty
       else "No L column has >50% missingness; full report saved.")
show_last()


Top-10 highest overall missingness:
vent_mode__hours_since_last__last_12h                    0.7550
lactate__last__last_12h                                  0.7159
po2_arterial__mean__last_12h                             0.6148
pco2_arterial__mean__last_12h                            0.6146
activated_partial_thromboplastin_time__last__last_12h    0.5323
fio2__last__last_12h                                     0.4536
hemoglobin__last__last_12h                               0.3465
fluid_out_urine__mean__last_12h                          0.3287
bicarbonate_arterial__last__last_12h                     0.2874
ureum__last__last_12h                                    0.2842

Missingness at t=0 (first 12h post-admit):
vent_mode__hours_since_last__last_12h                    0.7631
fio2__last__last_12h                                     0.5431
lactate__last__last_12h                                  0.4787
po2_arterial__mean__last_12h                             0.4751
pco2_arterial__mean__las

In [44]:
# G5: numeric confounder plausibility.
PLAUSIBLE_RANGE = {
    "heart_rate__mean__last_12h":            (20, 220),
    "arterial_blood_pressure_mean__mean__last_12h": (20, 200),
    "respiratory_rate_measured__mean__last_12h": (4, 60),
    "temperature__mean__last_12h":           (28, 43),
    "o2_saturation__mean__last_12h":         (50, 100),
    "po2_arterial__mean__last_12h":          (20, 600),
    "pco2_arterial__mean__last_12h":         (10, 120),
    "fluid_out_urine__mean__last_12h":       (0, 5000),
    "hemoglobin__last__last_12h":            (2, 25),
    "creatinine__last__last_12h":            (0, 2000),
    "ureum__last__last_12h":                 (0, 200),
    "lactate__last__last_12h":               (0, 30),
    "bicarbonate_arterial__last__last_12h":  (5, 50),
    "activated_partial_thromboplastin_time__last__last_12h": (15, 200),
}
issues = []
for col, (lo, hi) in PLAUSIBLE_RANGE.items():
    if col not in df.columns:
        continue
    s = df[col].dropna()
    n_below = int((s < lo).sum())
    n_above = int((s > hi).sum())
    if n_below + n_above > 0:
        issues.append({
            "col": col,
            "n_below": n_below, "n_above": n_above,
            "min": float(s.min()), "max": float(s.max()),
            "plausible_lo": lo, "plausible_hi": hi,
            "frac_outside": (n_below + n_above) / len(s),
        })

if issues:
    issues_df = pd.DataFrame(issues).sort_values("frac_outside", ascending=False)
    issues_df.to_csv(OUTPUT_DIR / "implausible_values.csv", index=False)
    print(issues_df.to_string(index=False))
else:
    print("All numeric confounders within plausible ranges.")

record("G.Distributional", "G5", "Numeric confounders within plausible clinical ranges",
       passed=len(issues) == 0,
       n_violations=len(issues),
       details=f"{len(issues)} columns out of range; see implausible_values.csv" if issues else "OK")
show_last()


                          col  n_below  n_above  min   max  plausible_lo  plausible_hi  frac_outside
pco2_arterial__mean__last_12h        0       27 10.0 147.0            10           120      0.000126
[FAIL] [G.Distributional/G5] Numeric confounders within plausible clinical ranges
        details: 1 columns out of range; see implausible_values.csv


In [45]:
# G6: vent-mode hours-since-last in [0, 12].
if VENT_HOURS_COL in df.columns:
    s = df[VENT_HOURS_COL].dropna()
    out_of_range = int(((s < 0) | (s > W_L_HOURS)).sum())
    record("G.Distributional", "G6", f"{VENT_HOURS_COL} in [0, {W_L_HOURS}]",
           passed=out_of_range == 0,
           n_violations=out_of_range,
           n_total=len(s),
           details=f"min={s.min():.2f}, max={s.max():.2f}, n_observed={len(s)}")
    show_last()


[PASS] [G.Distributional/G6] vent_mode__hours_since_last__last_12h in [0, 12] (0/136664)
        info: min=0.00, max=12.00, n_observed=136664


In [46]:
# G7: n_obs_in_window — zero counts and NaNs.
if "n_obs_in_window" in df.columns:
    n_obs = df["n_obs_in_window"]
    n_zero = int((n_obs == 0).sum())
    n_nan = int(n_obs.isna().sum())
    print(f"n_obs_in_window: zeros = {n_zero:,}, NaN = {n_nan:,}")
    print(n_obs.describe().round(2).to_string())
    record("G.Distributional", "G7", "n_obs_in_window distribution (informational)",
           passed=True,
           details=f"zeros: {n_zero}, NaN: {n_nan}, min: {n_obs.min()}, "
                   f"median: {n_obs.median()}, max: {n_obs.max()}")
    show_last()


n_obs_in_window: zeros = 0, NaN = 394
count    557502.00
mean         22.14
std           9.10
min           1.00
25%          17.00
50%          21.00
75%          26.00
max         713.00
[PASS] [G.Distributional/G7] n_obs_in_window distribution (informational)
        info: zeros: 0, NaN: 394, min: 1.0, median: 21.0, max: 713.0


In [47]:
# G8: categorical levels for last-vars.
levels_summary = {}
for col in LAST_VARS_CATEGORICAL:
    if col not in df.columns:
        continue
    vc = df[col].value_counts(dropna=False).head(20)
    levels_summary[col] = vc.to_dict()
    print(f"\n{col}:")
    print(vc.to_string())

with open(OUTPUT_DIR / "categorical_levels.json", "w") as f:
    # Convert NaN keys to "NaN" for JSON serialisation
    out = {col: {str(k): int(v) for k, v in d.items()}
           for col, d in levels_summary.items()}
    json.dump(out, f, indent=2)

record("G.Distributional", "G8", "Categorical level counts recorded (informational)",
       passed=True,
       details=f"Saved to {OUTPUT_DIR}/categorical_levels.json")
show_last()



glasgow_coma_scale_total__last__last_12h:
glasgow_coma_scale_total__last__last_12h
normal           219995
moderate         134328
severe           113886
mild_impaired     82762
NaN                6925

fio2__last__last_12h:
fio2__last__last_12h
NaN       253064
31_40     143503
41_50      81107
21_30      30555
51_60      19961
61_80      16646
81_100     13060

vent_mode__last__last_12h:
vent_mode__last__last_12h
unknown                421232
invasive_assisted       80432
invasive_controlled     55399
niv_assisted              772
cancelled                  61
[PASS] [G.Distributional/G8] Categorical level counts recorded (informational)
        info: Saved to validation_outputs/categorical_levels.json


In [48]:
# G9: in-ICU death timing by terminal t.
icu_death_term_full = term[(term["A"] == 0) & (term["D"] == 1)]
icu_death_t_dist = icu_death_term_full["t"].value_counts().sort_index()
print("In-ICU deaths by terminal t (first/last 10):")
print(pd.concat([icu_death_t_dist.head(10), icu_death_t_dist.tail(10)]).to_string())
icu_death_t_dist.to_csv(OUTPUT_DIR / "icu_death_terminal_t.csv")

# Information: are deaths concentrated at small t (likely sicker patients dying early)?
# This is informational; recorded.
median_death_t = icu_death_term_full["t"].median()
median_disch_t = disch_term["t"].median()
record("G.Distributional", "G9", "In-ICU death timing vs discharge timing (informational)",
       passed=True,
       details=f"median t at in-ICU death = {median_death_t}; "
               f"median t at discharge = {median_disch_t}")
show_last()


In-ICU deaths by terminal t (first/last 10):
t
0     196
1     473
2     536
3     390
4     352
5     268
6     276
7     212
8     231
9     172
50      8
51      9
52     11
53      4
54      7
55      5
56      9
57      6
58      3
59      4
[PASS] [G.Distributional/G9] In-ICU death timing vs discharge timing (informational)
        info: median t at in-ICU death = 7.0; median t at discharge = 2.0


## H. Compile and export reports

In [49]:
report_df = pd.DataFrame(report_rows)
report_df["timestamp"] = datetime.now().isoformat(timespec="seconds")

n_total = len(report_df)
n_pass  = int(report_df["passed"].sum())
n_fail  = n_total - n_pass

print("=" * 75)
print("VALIDATION SUMMARY")
print("=" * 75)
print(f"Total tests:    {n_total}")
print(f"Passed:         {n_pass}")
print(f"Failed:         {n_fail}")
print()
print("Per-category pass/fail:")
print(report_df.groupby("category")["passed"].agg(["sum", "count"]).to_string())
print()

if n_fail:
    print("FAILED TESTS:")
    cols = ["category", "test_id", "description", "n_violations", "n_total_checked", "details"]
    print(report_df[~report_df["passed"]][cols].to_string(index=False, max_colwidth=80))
else:
    print("All tests passed.")


VALIDATION SUMMARY
Total tests:    46
Passed:         44
Failed:         2

Per-category pass/fail:
                  sum  count
category                    
A.Schema            9      9
B.Grid              5      5
C.Cohort            3      3
D.Leakage           3      3
E.Encoding          9      9
F.Consistency       8      8
G.Distributional    7      9

FAILED TESTS:
        category test_id                                          description  n_violations  n_total_checked                                                                          details
G.Distributional      G4     L-column missingness recorded; flag >50% missing             5              NaN Columns with >50% missing: ['vent_mode__hours_since_last__last_12h', 'lactate...
G.Distributional      G5 Numeric confounders within plausible clinical ranges             1              NaN                               1 columns out of range; see implausible_values.csv


In [50]:
# Save outputs.
report_df.to_csv(OUTPUT_DIR / "validation_report.csv", index=False)
report_df.to_json(OUTPUT_DIR / "validation_report.json", orient="records", indent=2)

with open(OUTPUT_DIR / "validation_summary.txt", "w") as f:
    f.write(f"Validation summary -- {datetime.now().isoformat(timespec='seconds')}\n")
    f.write(f"Parquet: {PARQUET_PATH}\n")
    f.write(f"Rows: {len(df):,}, Stays: {df['stay_id'].nunique():,}\n\n")
    f.write(f"Total tests: {n_total}\n")
    f.write(f"Passed:      {n_pass}\n")
    f.write(f"Failed:      {n_fail}\n\n")
    f.write("Per-category breakdown:\n")
    f.write(report_df.groupby("category")["passed"].agg(["sum", "count"]).to_string())
    f.write("\n\n")
    if n_fail:
        f.write("FAILED TESTS:\n")
        for _, r in report_df[~report_df["passed"]].iterrows():
            f.write(f"  [{r['category']}/{r['test_id']}] {r['description']}\n")
            f.write(f"     violations: {r['n_violations']} / {r['n_total_checked']}\n")
            if r['details']:
                f.write(f"     details: {r['details']}\n")
            if r['sample_violating_stay_ids']:
                f.write(f"     sample: {r['sample_violating_stay_ids']}\n")
            f.write("\n")

print(f"\nReports saved to: {OUTPUT_DIR}/")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size:,} bytes)")



Reports saved to: validation_outputs/
  F3_candidates_admin_with_death_in_window.csv  (503 bytes)
  categorical_levels.json  (512 bytes)
  discharge_rate_per_t.csv  (1,375 bytes)
  icu_death_terminal_t.csv  (365 bytes)
  implausible_values.csv  (143 bytes)
  label_vs_derived_icu_mortality.csv  (68 bytes)
  missingness_by_t.csv  (20,949 bytes)
  missingness_overall.csv  (921 bytes)
  rows_per_t.csv  (478 bytes)
  validation_report.csv  (6,016 bytes)
  validation_report.json  (16,024 bytes)
  validation_summary.txt  (1,064 bytes)


In [51]:
# Display the full report inline.
pd.set_option("display.max_colwidth", 100)
report_df[["category", "test_id", "description", "passed",
           "n_violations", "n_total_checked", "violation_rate"]]


,category,test_id,description,passed,n_violations,n_total_checked,violation_rate
0,A.Schema,A1,All required columns present,True,0,NaN,NaN
1,A.Schema,A2,"(stay_id, t) is a unique key",True,0,557896.0,0.000000
2,A.Schema,A2b,"(stay_id, k) is a unique key",True,0,557896.0,0.000000
3,A.Schema,A3,"No NaN in stay_id, t, k, grid_end, intime, outtime, los",True,0,NaN,NaN
4,A.Schema,A4,"t in [0, 59] and k in [1, 60]",True,0,557896.0,0.000000
5,A.Schema,A4b,k == t + 1 for every row,True,0,557896.0,0.000000
6,A.Schema,A5,"A in {0,1}; D, Z, Y in {0, 1, NaN}",True,0,NaN,NaN
7,A.Schema,A6,A is never NaN,True,0,557896.0,0.000000
8,A.Schema,A7,Column dtypes match expectations,True,0,NaN,NaN
9,B.Grid,B1,Per-stay t is contiguous starting at 0,True,0,81736.0,0.000000


## How to share results

After running the notebook, share these files for analysis of any failures:

1. **`validation_outputs/validation_report.csv`** — structured pass/fail rows
2. **`validation_outputs/validation_summary.txt`** — human-readable summary
3. **`validation_outputs/missingness_overall.csv`** and **`missingness_by_t.csv`** — for category G context
4. **`validation_outputs/implausible_values.csv`** (if non-empty) — for G5 follow-up
5. **`validation_outputs/label_vs_derived_icu_mortality.csv`** — for F5

## Interpreting failures by category

- **A, B, E** — **hard contract violations**. Must be resolved before any analysis. Indicate schema or encoding bugs.
- **C** — **eligibility leaks**. Indicate upstream filter regressions that change the cohort definition.
- **D** — **leakage**. Reverse-causation contamination of $L_t$ from discharge information.
- **F** — **internal inconsistency**. Usually a definitional question (e.g., label vs derived).
- **G** — **distributional / data-quality**. Often informational rather than blocking; failures should prompt follow-up rather than panic.

When you encounter a failure, share the report file and the **`sample_violating_stay_ids`** from the failing row(s). For any failed test, the corresponding stay_ids can be reloaded and inspected directly to understand what went wrong.


# I ran it myself. Here is the report. Can I reasonably move to running on pygformula with these violations noted? Or are they critical points that need to be worked on?

# Verdict: yes, you can proceed — but with eyes open about G4

## What the report says (and what it means for pygformula)

Every hard-contract category passes:

- **A.Schema (9/9)** — column dtypes, key uniqueness, value domains. pygformula needs all of this to be clean to construct its internal data structures.
- **B.Grid (5/5)** — $t$ contiguity, grid arithmetic. pygformula assumes contiguous time indices; you have them.
- **C.Cohort (3/3)** — eligibility filters held. The cohort matches the estimand.
- **D.Leakage (3/3)** — 5h pre-discharge buffer respected. No reverse-causation contamination of $L_t$.
- **E.Encoding (9/9)** — the $(A, D, Z, Y)$ contract is intact. pygformula's competing-event handling depends on this exactly.
- **F.Consistency (8/8)** — including F3, which surfaced two encoder edge cases that I'd flagged earlier; these are documented as informational, not bugs.

The remaining two failures (G4, G5) are in the informational category and are characteristic of MIMIC-IV ICU data, not artifacts of your pipeline. Let me address each.

## G5 (single column out of plausible range)

`pco2_arterial` had 27 values above 120 (0.012% of observed). This is **noise-level**; the values are physiologically extreme but not impossible (severe acute respiratory failure can produce $\text{PaCO}_2 > 120$). Two options:

- Leave as-is and document in methods.
- Tighten clipping in `3_organize`'s `CLIP_RULES` for `pco2_arterial` to e.g. $[10, 130]$ as a sensitivity analysis.

Either is defensible. Not a blocker.

## G4 (five columns with >50% missingness) — the only thing that warrants real attention

This is the one finding that **does** have analytic consequences for pygformula. Let me work through it carefully, because the implications depend on which variable.

The five columns and their missingness mechanisms:

| Column | Likely missingness mechanism | Impact on g-formula |
|---|---|---|
| `vent_mode__hours_since_last__last_12h` | 75.5% missing — almost certainly **MAR conditional on vent_mode != "unknown"**. The hours-since-last is undefined for patients not on a ventilator (which is most of the cohort). | **Low**. The structural pattern (NaN ⇔ no vent mode known) is informative and can be encoded as a missingness indicator |
| `lactate__last__last_12h` | ~71% missing — lactate is **only drawn when clinically indicated** (suspected sepsis, shock). Missingness is informative: "no lactate measured" ≈ "doctor wasn't worried about lactate." | **Higher**. MNAR. Treating NaN as "value missing at random" biases the propensity model |
| `pco2_arterial__mean__last_12h` | ~61% missing — ABG only done when intubated or on high-flow O2 | MNAR, conditional on ventilation status |
| `po2_arterial__mean__last_12h` | ~61% missing — same as above (ABG-conditioned) | MNAR, conditional on ventilation status |
| `activated_partial_thromboplastin_time__last__last_12h` | ~53% missing — coag panel only ordered on bleeding/anticoag suspicion | MNAR |

## The substantive issue

pygformula doesn't automatically handle missingness — it expects clean inputs. Three options, in increasing rigor:

### Option 1: LGBM handles NaN natively, GLM does not (your nuisance choice matters)

You moved to LGBM nuisance models, which is fortunate for this exact reason. LightGBM treats NaN as a separate split direction during tree fitting — it learns "when this column is missing, take this branch." This is implicitly **a missingness-indicator model**, applied at every split. For >50%-missing columns, this is generally adequate and often better than imputation.

**Implication for pygformula**: if you're using LGBM as the conditional-density estimator for the $L_t \mid \bar{L}_{t-1}, \bar{A}_{t-1}$ models, the missingness is structurally handled. No additional imputation needed. The cost is interpretability — the propensity scores at large $t$ depend on missingness patterns in a way that's hard to inspect.

### Option 2: Explicit missingness indicators + imputation

For each of the five columns, add a binary `_missing` indicator and impute the missing values with a fixed value (e.g., median, or "physiological normal"). This makes the missingness mechanism explicit in the model and is robust under MAR-given-indicator.

```python
for col in ['lactate__last__last_12h', 'pco2_arterial__mean__last_12h',
            'po2_arterial__mean__last_12h',
            'activated_partial_thromboplastin_time__last__last_12h']:
    df[f"{col}__missing"] = df[col].isna().astype(int)
    df[col] = df[col].fillna(df[col].median())
```

This is the **standard MAR-given-observed approach** and is what most causal-inference literature does when missingness is suspected to be informative. Reviewer-defensible.

For `vent_mode__hours_since_last`, the structural rule is different: NaN means "no vent recorded recently," so imputing with $w_L = 12$h (the maximum, meaning "vent record is old or absent") plus an indicator is cleaner than the median.

### Option 3: Drop the worst offenders from the L set

If `lactate` or `aPTT` is contributing little to the propensity model's predictive performance, drop them. Run a quick LGBM feature-importance check on $A_t \mid L_t$ at $t = 0$:

```python
from lightgbm import LGBMClassifier
import lightgbm as lgb

t0 = df[df['t'] == 0].dropna(subset=['A'])
X = t0[ALL_L_COLS].copy()
y = t0['A']
model = LGBMClassifier(n_estimators=200, num_leaves=31).fit(X, y)
imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp)
```

If `lactate` or `aPTT` are at the bottom and their missingness is hurting more than their signal helps, drop them and rerun.

## My recommendation

**Proceed to pygformula now**, using Option 1 (LGBM handles NaN natively). Do the analysis with the data as-is. **Then** as a sensitivity analysis, implement Option 2 (indicators + imputation) and report whether the headline estimate moves materially. This is the pattern most causal-inference papers follow, and matches your diagnostic-first framing.

Two things to add to the methods text:

1. **Disclose the missingness rates** for the five MNAR-suspect columns. Reviewers will check.
2. **Justify LGBM's NaN handling** as the implicit missingness-indicator model. Cite Chen & Guestrin (XGBoost paper, 2016) for the formal treatment of NaN as a learned split direction. This pre-empts the "did you handle missingness?" reviewer question.

## What I would NOT do

- **Don't impute lactate via mean/median without an indicator.** Reviewers know lactate is ordered for sepsis suspicion; pretending the missing values are "normal" biases the propensity model toward treating sick patients as well.
- **Don't drop rows with NaN.** That's listwise deletion, which biases by construction when missingness is MNAR — and it would gut your cohort.
- **Don't switch back to GLM nuisance models** just to "see if it changes things." GLM requires you to solve the missingness problem first; with LGBM you can defer it.

## On the two F3 edge cases (informational)

The two `mimic4-39353880` and `mimic4-39463279` cases — where death occurred before $K$ but the encoder classified them as admin-censored — are 2 / 81,736 = 0.0024% of stays. The g-formula estimate's standard error is going to be several orders of magnitude larger than the effect of these two misclassifications. Note them in a methods appendix paragraph; do not block on them.

If you want to be thorough, you could patch the `5_organize` terminal-row encoder to check `death_time_from_intime <= K_HOURS` and force the in-ICU-death classification regardless of LOS. But the impact on results will be undetectable.

## Bottom line

**Go.** The hard contracts are intact, the encoding is clean, and the missingness is a known property of ICU data that your nuisance model (LGBM) can handle natively. Pre-specify the missingness-indicator sensitivity analysis in your methods, mention G4 explicitly in the limitations, and run pygformula.

Want me to write the missingness-sensitivity preprocessing block for Option 2, so it's ready when you need it for the sensitivity comparison?